In [11]:
import os
import pandas as pd
import numpy as np

DATA_PATH = "../data/raw/train-00000-of-00001.parquet"

# Verifikasi keberadaan file dan periksa ukuran fisik (MB)
if os.path.exists(DATA_PATH):
    file_size_mb = os.path.getsize(DATA_PATH) / (1024 * 1024)
    print(f"Dataset ditemukan: {DATA_PATH}")
    print(f"Ukuran File: {file_size_mb:.2f} MB")
else:
    print(f"File tidak ditemukan di {DATA_PATH}. Pastikan file sudah diletakkan di folder data/raw/")

Dataset ditemukan: ../data/raw/train-00000-of-00001.parquet
Ukuran File: 165.88 MB


In [13]:
df = pd.read_parquet(DATA_PATH)

print("=== DAFTAR KOLOM DAN TIPE DATA AWAL ===")
print(df.dtypes)

print("\n=== JUMLAH BARIS ===")
print(df.shape)

print("\n=== 5 BARIS PERTAMA ===")
df.head()

=== DAFTAR KOLOM DAN TIPE DATA AWAL ===
text          str
label_text    str
source        str
split         str
dtype: object

=== JUMLAH BARIS ===
(1030393, 4)

=== 5 BARIS PERTAMA ===


,text,label_text,source,split
0,warung ini dimiliki oleh pengusaha pabrik tahu...,positive,indonlp/indonlu/smsa,train
1,mohon ulama lurus dan k212 mmbri hujjah partai...,neutral,indonlp/indonlu/smsa,train
2,lokasi strategis di jalan sumatera bandung . t...,positive,indonlp/indonlu/smsa,train
3,betapa bahagia nya diri ini saat unboxing pake...,positive,indonlp/indonlu/smsa,train
4,duh . jadi mahasiswa jangan sombong dong . kas...,negative,indonlp/indonlu/smsa,train


In [15]:
import pyarrow.parquet as pq

# Penghitungan Dimensi & Profiling Missing Values
CHUNK_SIZE = 100_000
total_rows = 0
null_counts = None

pf = pq.ParquetFile(DATA_PATH)

for batch in pf.iter_batches(batch_size=CHUNK_SIZE):
    chunk = batch.to_pandas()
    total_rows += len(chunk)
    if null_counts is None:
        null_counts = chunk.isnull().sum()
    else:
        null_counts += chunk.isnull().sum()

print(f"Total Baris Data : {total_rows:,}")
print(f"Total Kolom Data : {len(null_counts)}")

# Membuat tabel ringkasan missing values
null_summary = pd.DataFrame({
    'Missing_Values': null_counts,
    'Percentage (%)': (null_counts / total_rows * 100).round(2)
})
print("\n=== RINGKASAN MISSING VALUES ===")
null_summary

Total Baris Data : 1,030,393
Total Kolom Data : 4

=== RINGKASAN MISSING VALUES ===


,Missing_Values,Percentage (%)
text,0,0.0
label_text,0,0.0
source,0,0.0
split,0,0.0


In [16]:
import pyarrow.parquet as pq

pf = pq.ParquetFile(DATA_PATH)
df_profile = next(pf.iter_batches(batch_size=200_000)).to_pandas()

print("=== STATISTIK KOLOM NUMERIK ===")
num_cols = df_profile.select_dtypes(include=[np.number]).columns
if len(num_cols) > 0:
    display(df_profile[num_cols].describe())
else:
    print("Tidak ada kolom numerik.")

print("\n=== STATISTIK KOLOM KATEGORIKAL ===")
cat_cols = df_profile.select_dtypes(include=['object', 'string']).columns
if len(cat_cols) > 0:
    display(df_profile[cat_cols].describe())
else:
    print("Tidak ada kolom kategorikal.")

=== STATISTIK KOLOM NUMERIK ===
Tidak ada kolom numerik.

=== STATISTIK KOLOM KATEGORIKAL ===


,text,label_text,source,split
count,200000,200000,200000,200000
unique,199890,3,2,3
top,0,positive,FDR Dataset,train
freq,108,138191,187321,198254


In [17]:
duplicates = df_profile.duplicated().sum()
print(f"Jumlah baris duplikat pada sampel 200k: {duplicates:,} ({(duplicates / len(df_profile)) * 100:.2f}%)")

Jumlah baris duplikat pada sampel 200k: 108 (0.05%)


In [19]:
TEXT_COL = "text"
LABEL_COL = "label_text"

print("=== JUMLAH NILAI UNIK TIAP KOLOM ===")
display(df_profile.nunique())

# Distribusi label
if LABEL_COL in df_profile.columns:
    print("\n=== DISTRIBUSI LABEL ===")
    display(df_profile[LABEL_COL].value_counts())
    display(df_profile[LABEL_COL].value_counts(normalize=True).round(3))

# Distribusi sumber dan split
for col in ["source", "split"]:
    if col in df_profile.columns:
        print(f"\n=== DISTRIBUSI {col.upper()} ===")
        display(df_profile[col].value_counts())

# Panjang teks
if TEXT_COL in df_profile.columns:
    print("\n=== PANJANG TEKS (jumlah karakter) ===")
    display(df_profile[TEXT_COL].str.len().describe())
    print("\nContoh teks:", df_profile[TEXT_COL].iloc[0])

=== JUMLAH NILAI UNIK TIAP KOLOM ===


text          199890
label_text         3
source             2
split              3
dtype: int64


=== DISTRIBUSI LABEL ===


label_text
positive    138191
neutral      35725
negative     26084
Name: count, dtype: int64

label_text
positive    0.691
neutral     0.179
negative    0.130
Name: proportion, dtype: float64

...


=== DISTRIBUSI SPLIT ===


split
train         198254
validation      1246
test             500
Name: count, dtype: int64


=== PANJANG TEKS (jumlah karakter) ===


count    200000.000000
mean        343.900125
std         188.608271
min           1.000000
25%         233.000000
50%         291.000000
75%         397.000000
max        8745.000000
Name: text, dtype: float64


Contoh teks: warung ini dimiliki oleh pengusaha pabrik tahu yang sudah puluhan tahun terkenal membuat tahu putih di bandung . tahu berkualitas , dipadu keahlian memasak , dipadu kretivitas , jadilah warung yang menyajikan menu utama berbahan tahu , ditambah menu umum lain seperti ayam . semuanya selera indonesia . harga cukup terjangkau . jangan lewatkan tahu bletoka nya , tidak kalah dengan yang asli dari tegal !
